# Workflow

This notebook illustrates how to create training data, and train the various modules (rescaler, sbi, unfolder) on the local machine.

See `htcondor_workflow.ipynb` for generating the training set with the HTCondor scheduler instead.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False);

from labrador import (
    compression,
    generate_parameters,
    rescaling,
    simulation,
    training,
    unfolding,
    utils,
    weighting,
)

## 1. Set up run directory and configuration file
Let us make a run directory with a copy of the example `data_config.py` in it.

After this you can edit the new config file as needed.

In [ ]:
parentdir = '../scratch/'  # Edit as appropriate

rundir = utils.setup_rundir(parentdir)  # You may then edit the contents of the new data_config file

Note: alternatively this can be run on a terminal like so:

    lab-setup-rundir {parentdir}

## 2. Generate simulation parameters

This will produce a dataframe `simulation_parameters.feather` with binary black hole parameters.

In [ ]:
generate_parameters.main(rundir)

Note: alternatively this could be run on a terminal like so:

    export OMP_NUM_THREADS=1
    python -m labrador.generate_parameters {rundir}

## 3. Simulate signals and preprocess the data

This will produce directories for the training and test datasets, containing the following files:
* `preprocessed_data.npz`: Auxiliary file with heterodyned data, heterodyned signals, and phenomenological reference waveform parameters.
* `folded_sampled_params.npy`: Contains the true (injected) parameter values, after folding.
* `unfolding_labels.npy`: Contains the true (injected) index of the unfolding transformation. 

In [ ]:
simulation.main(rundir, processes=None)

Note: alternatively this could be run on the terminal as

    export OMP_NUM_THREADS=1
    python -m labrador.simulation {rundir}

## 4. Compress the data

This will create `compressed_data.npy`, which should be suitable for training the network.

In [ ]:
compression.create_mask(rundir)
compression.svd_compression(rundir)

Note: The compression is implemented as a step separate from preprocessing for the practical reason that the preprocessing is assumed to be trivially parallelizable (each preprocessed data can be computed in isolation from the other simulations).
On the other hand, compression algorithms might need to learn the distribution of the preprocessed data.

## 5. Compute the prior weights
We generated simulations from an unphysical simulation prior. In order for the training to produce posteriors under a physical prior, we need to reweight these simulations.

In [ ]:
weighting.main(rundir)

### File tree
So far we have created these training and test data:

In [ ]:
! tree --filesfirst {rundir}

## 6. Rescale the parameters
Now we train a neural network to predict the mean and variance of the posterior, and rescale our coordinates using these estimates.

In [ ]:
priordir = utils.get_priordirs(rundir)[0]
rescalerdir = utils.setup_rescalerdir(priordir)

Edit the config file if needed. Then, preferably in a machine with a GPU, run:

In [ ]:
rescaling.main(rescalerdir)

or equivalently

    python -m labrador.rescaling {rescalerdir}

## 7. Train the neural posterior estimator and unfolder
*7a and 7b are independent of each other, they can happen in parallel. 7b is typically fast.*

### 7a. Neural posterior estimator

First, set up an `sbidir` with its own `sbi_config.py` file:

In [ ]:
sbidir = utils.setup_sbidir(rescalerdir)  # You may then edit the contents of the new config file

Again, edit the new config file as needed.

Then, preferably in a machine with a GPU, run:

In [ ]:
training.main(sbidir)

or equivalently

    python -m labrador.training {sbidir}

These last steps can be repeated as needed to train multiple models (e.g. varying the network architecture) without having to regenerate the data.

### 7b. Unfolding classifier

In [ ]:
unfolderdir = utils.setup_unfolderdir(rescalerdir)

You may edit the config file. Then,

In [ ]:
unfolding.main(unfolderdir)

or equivalently

    python -m labrador.unfolding {unfolderdir}


#### Plot confusion matrix

It should look the same on the training and test sets, else there's overfitting.
Note, in general it should not be a diagonal matrix.

In [ ]:
unfolder = unfolding.UnfoldingClassifier(unfolderdir)
unfolder.plot_confusion_matrix()

Congrats, you have trained your `labrador`!